# Project 07: Core NLP Spam Classification (NeuroGebra Native)

Build a practical spam detector with TF-IDF features and NeuroGebra ModelBuilder.


## Dataset Source and Download Instructions

- Dataset: SMS Spam Collection (UCI Machine Learning Repository)
- Official source: https://archive.ics.uci.edu/dataset/228/sms+spam+collection
- License: see UCI dataset terms
- Auto-download in this notebook: URL download from UCI static asset
- Manual fallback:
  1. Download `sms+spam+collection.zip` from UCI.
  2. Extract `SMSSpamCollection` into `./data/sms_spam/`.
  3. Re-run parsing and training cells.


In [ ]:
%pip -q install neurogebra pandas scikit-learn matplotlib


In [ ]:
import random
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from neurogebra.builders.model_builder import ModelBuilder

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
data_dir = Path("./data/sms_spam")
data_dir.mkdir(parents=True, exist_ok=True)
zip_path = data_dir / "sms_spam_collection.zip"

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
if not zip_path.exists():
    urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, "r") as zf:
    with zf.open("SMSSpamCollection") as f:
        rows = [line.decode("utf-8", "ignore").strip().split("	", 1) for line in f.readlines()]

df = pd.DataFrame(rows, columns=["label", "text"])
df["target"] = (df["label"] == "spam").astype(np.float64)

print(df.head())
print(df["target"].value_counts(normalize=True))


In [ ]:
vectorizer = TfidfVectorizer(max_features=6000, ngram_range=(1, 2), min_df=2)
X = vectorizer.fit_transform(df["text"]).astype(np.float64).toarray()
y = df["target"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

builder = ModelBuilder()
model = builder.Sequential(
    [
        builder.Dense(256, activation="relu", input_shape=(X_train.shape[1],)),
        builder.Dropout(0.3),
        builder.Dense(64, activation="relu"),
        builder.Dense(1, activation="sigmoid"),
    ],
    name="sms_spam_classifier",
)

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    learning_rate=0.001,
    log_level="detailed",
)

history = model.fit(
    X_train,
    y_train,
    epochs=8,
    batch_size=64,
    validation_split=0.1,
)


In [ ]:
probs = model.predict(X_test).reshape(-1)
preds = (probs > 0.5).astype(int)

print(classification_report(y_test.astype(int), preds, digits=3))
print("Confusion matrix:\n", confusion_matrix(y_test.astype(int), preds))

plt.figure(figsize=(8, 4))
plt.plot(history["loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.title("Spam Classifier Loss")
plt.legend()
plt.show()
